In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

TSTR_ROOT = Path("results/tstr")
OUTDIR = Path("results/fmsd_tables")
OUTDIR.mkdir(parents=True, exist_ok=True)

MODELS = ["moirai", "chronos"]
SINGLE_SEED = 42
COMPOSITION_SEEDS = [42, 43, 44]

CRPS_COL = "mean_weighted_sum_quantile_loss"
MASE_COL = "MASE[0.5]"

SINGLE_GENERATOR_CONDITIONS = {
    "arima": "arima",
    "fbm": "fbm",
    "waveform": "waveform",
    "chaotic": "chaotic",
    "ets": "ets",
    "garch": "garch",
    "kernelsynth": "kernelsynth",
    "sde": "sde",
    "stepfunction": "stepfunction",
    "timesynth": "timesynth",
    "tsi": "tsi",
}

ANCHOR_CONDITIONS = {
    "real_reference": "Real Ref",
    "mixed_all11": "Mixed",
}

COMPOSITION_CONDITIONS = {
    "real_reference": "Real Ref",
    "mixed_real_7525": "75-25",
    "mixed_real_5050": "50-50",
    "mixed_real_2575": "25-75",
    "mixed_all11": "Mixed",
}

MIXTURE_RATIO_CONDITIONS = [
    "mixed_real_7525",
    "mixed_real_5050",
    "mixed_real_2575",
]

HORIZON_ORDER = ["short", "medium", "long"]
COMPOSITION_ORDER = ["Real Ref", "75-25", "50-50", "25-75", "Mixed"]

baseline_path = TSTR_ROOT / "seasonal_naive" / "all_results.csv"
baseline = pd.read_csv(baseline_path)
baseline = baseline[["dataset", CRPS_COL, MASE_COL]].rename(columns={
    CRPS_COL: "baseline_crps",
    MASE_COL: "baseline_mase",
})


def gmean_positive(x):
    x = pd.Series(x).replace([np.inf, -np.inf], np.nan).dropna()
    x = x[x > 0]
    if len(x) == 0:
        return np.nan
    return float(np.exp(np.log(x).mean()))


def parse_horizon(dataset_name):
    parts = str(dataset_name).split("/")
    return parts[-1] if len(parts) >= 2 else "unknown"


def load_condition(model, seed, condition, label, run_type):
    path = TSTR_ROOT / model / f"seed_{seed}" / condition / "all_results.csv"
    if not path.exists():
        print(f"missing: {model} seed={seed} condition={condition}")
        return None

    df = pd.read_csv(path)
    required = ["dataset", CRPS_COL, MASE_COL, "domain"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{path} missing columns: {missing}")

    df = df.merge(baseline, on="dataset", how="inner")
    df = df.replace([np.inf, -np.inf], np.nan)

    df["model_family"] = model
    df["seed"] = seed
    df["condition"] = condition
    df["label"] = label
    df["run_type"] = run_type
    df["horizon"] = df["dataset"].apply(parse_horizon)

    df["norm_crps"] = df[CRPS_COL] / df["baseline_crps"]
    df["norm_mase"] = df[MASE_COL] / df["baseline_mase"]

    return df


def condition_scores(task_df, group_cols):
    return (
        task_df
        .groupby(group_cols, as_index=False)
        .agg(
            n_tasks=("dataset", "nunique"),
            norm_crps=("norm_crps", gmean_positive),
            norm_mase=("norm_mase", gmean_positive),
        )
    )


def add_ranks(df, group_cols=("model_family",), crps_col="norm_crps", mase_col="norm_mase"):
    df = df.copy()
    df["rank_crps"] = df.groupby(list(group_cols))[crps_col].rank(ascending=True, method="min")
    df["rank_mase"] = df.groupby(list(group_cols))[mase_col].rank(ascending=True, method="min")
    df["rank_mean"] = df[["rank_crps", "rank_mase"]].mean(axis=1)
    return df


def fmt(x):
    if pd.isna(x):
        return "--"
    return f"{x:.2f}"


def fmt_mean_std(mean, std, is_best=False):
    if pd.isna(mean):
        return "--"
    std = 0.0 if pd.isna(std) else std
    text = f"{mean:.2f} ± {std:.2f}"
    return f"★ {text}" if is_best else text

In [2]:
# Load seed-42 single-generator runs plus seed-42 anchors.
single_rows = []

for model in MODELS:
    for condition, generator in SINGLE_GENERATOR_CONDITIONS.items():
        df = load_condition(model, SINGLE_SEED, condition, generator, "single_generator")
        if df is not None:
            single_rows.append(df)

    for condition, label in ANCHOR_CONDITIONS.items():
        df = load_condition(model, SINGLE_SEED, condition, label, "anchor")
        if df is not None:
            single_rows.append(df)

single_tasks = pd.concat(single_rows, ignore_index=True)
print(single_tasks[["model_family", "seed", "condition", "label", "run_type"]].drop_duplicates().sort_values(["model_family", "run_type", "label"]).to_string(index=False))

model_family  seed       condition        label         run_type
     chronos    42     mixed_all11        Mixed           anchor
     chronos    42  real_reference     Real Ref           anchor
     chronos    42           arima        arima single_generator
     chronos    42         chaotic      chaotic single_generator
     chronos    42      fringe_ets          ets single_generator
     chronos    42    baseline_fbm          fbm single_generator
     chronos    42           garch        garch single_generator
     chronos    42     kernelsynth  kernelsynth single_generator
     chronos    42             sde          sde single_generator
     chronos    42    stepfunction stepfunction single_generator
     chronos    42       timesynth    timesynth single_generator
     chronos    42             tsi          tsi single_generator
     chronos    42 center_waveform     waveform single_generator
      moirai    42     mixed_all11        Mixed           anchor
      moirai    42  real_

In [3]:
# Main ranking: 11 generators + Real Ref only. Mixed is shown later as an anchor, not ranked as a single generator.
single_ranking_tasks = single_tasks[
    (single_tasks["run_type"] == "single_generator") |
    (single_tasks["condition"] == "real_reference")
].copy()

single_rank_table = condition_scores(
    single_ranking_tasks,
    ["model_family", "run_type", "condition", "label"],
)
single_rank_table = add_ranks(single_rank_table)
single_rank_table = single_rank_table.sort_values(["model_family", "norm_crps"])

display(single_rank_table)
single_rank_table.to_csv(OUTDIR / "seed42_single_generator_ranking_with_real.csv", index=False)

,model_family,run_type,condition,label,n_tasks,norm_crps,norm_mase,rank_crps,rank_mase,rank_mean
0,chronos,anchor,real_reference,Real Ref,97,0.791115,1.061018,1.0,1.0,1.0
7,chronos,single_generator,kernelsynth,kernelsynth,97,0.936023,1.290165,2.0,3.0,2.5
8,chronos,single_generator,sde,sde,97,0.980541,1.282208,3.0,2.0,2.5
9,chronos,single_generator,stepfunction,stepfunction,97,1.048485,1.358900,4.0,4.0,4.0
6,chronos,single_generator,garch,garch,97,1.107920,1.862916,5.0,7.0,6.0
1,chronos,single_generator,arima,arima,97,1.127727,1.915701,6.0,9.0,7.5
10,chronos,single_generator,timesynth,timesynth,97,1.152862,1.883684,7.0,8.0,7.5
11,chronos,single_generator,tsi,tsi,97,1.195281,1.679096,8.0,5.0,6.5
3,chronos,single_generator,center_waveform,waveform,97,1.327756,1.840838,9.0,6.0,7.5
2,chronos,single_generator,baseline_fbm,fbm,97,1.681219,2.494743,10.0,10.0,10.0


In [4]:
# Top-3 generators per model, plus Mixed and Real Ref anchors for direct reading.
generator_rank_table = single_rank_table[single_rank_table["run_type"] == "single_generator"].copy()
top3_generators = (
    generator_rank_table
    .sort_values(["model_family", "rank_mean", "norm_crps"])
    .groupby("model_family", as_index=False)
    .head(3)
)

context_scores = condition_scores(
    single_tasks,
    ["model_family", "run_type", "condition", "label"],
)

rows = []
for model in MODELS:
    keep_conditions = set(top3_generators.loc[top3_generators["model_family"] == model, "condition"])
    keep_conditions.update(["mixed_all11", "real_reference"])
    rows.append(context_scores[
        (context_scores["model_family"] == model) &
        (context_scores["condition"].isin(keep_conditions))
    ])

top3_vs_anchors_table = pd.concat(rows, ignore_index=True)
top3_vs_anchors_table = top3_vs_anchors_table.sort_values(["model_family", "norm_crps", "norm_mase"])

display(top3_vs_anchors_table)
top3_vs_anchors_table.to_csv(OUTDIR / "seed42_top3_generators_vs_mixed_and_real.csv", index=False)

,model_family,run_type,condition,label,n_tasks,norm_crps,norm_mase
6,chronos,anchor,real_reference,Real Ref,97,0.791115,1.061018
5,chronos,anchor,mixed_all11,Mixed,97,0.906495,1.171138
7,chronos,single_generator,kernelsynth,kernelsynth,97,0.936023,1.290165
8,chronos,single_generator,sde,sde,97,0.980541,1.282208
9,chronos,single_generator,stepfunction,stepfunction,97,1.048485,1.358900
4,moirai,single_generator,kernelsynth,kernelsynth,97,0.733835,1.048953
0,moirai,anchor,mixed_all11,Mixed,97,0.735103,1.069077
1,moirai,anchor,real_reference,Real Ref,97,0.814336,1.148936
3,moirai,single_generator,fringe_ets,ets,97,0.820390,1.153630
2,moirai,single_generator,center_waveform,waveform,97,0.869929,1.240054


In [5]:
def paired_bootstrap_compare(task_df, model, condition_a, condition_b, metric, pair_cols, n_boot=10_000, seed=42):
    a = (
        task_df[(task_df["model_family"] == model) & (task_df["condition"] == condition_a)]
        [list(pair_cols) + [metric]]
        .rename(columns={metric: "a"})
    )
    b = (
        task_df[(task_df["model_family"] == model) & (task_df["condition"] == condition_b)]
        [list(pair_cols) + [metric]]
        .rename(columns={metric: "b"})
    )

    paired = a.merge(b, on=list(pair_cols), how="inner")
    paired = paired.replace([np.inf, -np.inf], np.nan).dropna()
    paired = paired[(paired["a"] > 0) & (paired["b"] > 0)]
    if paired.empty:
        raise ValueError(f"No paired rows for {model}: {condition_a} vs {condition_b} on {metric}")

    d = np.log(paired["a"].to_numpy()) - np.log(paired["b"].to_numpy())
    n = len(d)
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot)

    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot[i] = np.exp(d[idx].mean()) - 1

    rel_delta = np.exp(d.mean()) - 1
    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

    label_lookup = (
        task_df[["model_family", "condition", "label"]]
        .drop_duplicates()
        .set_index(["model_family", "condition"])["label"]
        .to_dict()
    )

    return {
        "model_family": model,
        "condition_a": condition_a,
        "label_a": label_lookup.get((model, condition_a), condition_a),
        "condition_b": condition_b,
        "label_b": label_lookup.get((model, condition_b), condition_b),
        "metric": metric,
        "n_pairs": n,
        "rel_delta": rel_delta,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_a_better": float((boot < 0).mean()),
        "win_rate": float((d < 0).mean()),
    }


def format_bootstrap_table(df):
    out = df.copy()
    out["Rel. delta"] = out["rel_delta"].map(lambda x: f"{100*x:.1f}%")
    out["95% CI"] = out.apply(lambda r: f"[{100*r['ci_low']:.1f}%, {100*r['ci_high']:.1f}%]", axis=1)
    out["P(a better)"] = out["p_a_better"].map(lambda x: f"{x:.2f}")
    out["Win rate"] = out["win_rate"].map(lambda x: f"{x:.2f}")
    return out[[
        "model_family", "label_a", "label_b", "metric", "n_pairs",
        "Rel. delta", "95% CI", "P(a better)", "Win rate",
    ]]

In [6]:
# Seed-42 bootstrap: best vs second-best generator, best vs Mixed, best vs Real Ref.
single_boot_rows = []

for model in MODELS:
    ranked = generator_rank_table[generator_rank_table["model_family"] == model].sort_values(["rank_mean", "norm_crps"])
    if len(ranked) < 2:
        continue

    best = ranked.iloc[0]["condition"]
    second = ranked.iloc[1]["condition"]

    comparisons = [
        (best, second),
        (best, "mixed_all11"),
        (best, "real_reference"),
    ]

    for condition_a, condition_b in comparisons:
        for metric in ["norm_crps", "norm_mase"]:
            single_boot_rows.append(
                paired_bootstrap_compare(
                    single_tasks,
                    model=model,
                    condition_a=condition_a,
                    condition_b=condition_b,
                    metric=metric,
                    pair_cols=("dataset",),
                    n_boot=10_000,
                    seed=42,
                )
            )

single_bootstrap = pd.DataFrame(single_boot_rows)
single_bootstrap_paper = format_bootstrap_table(single_bootstrap)

display(single_bootstrap_paper)
single_bootstrap.to_csv(OUTDIR / "seed42_single_bootstrap_raw.csv", index=False)
single_bootstrap_paper.to_csv(OUTDIR / "seed42_single_bootstrap_table.csv", index=False)

,model_family,label_a,label_b,metric,n_pairs,Rel. delta,95% CI,P(a better),Win rate
0,moirai,kernelsynth,ets,norm_crps,97,-10.6%,"[-14.9%, -6.3%]",1.00,0.62
1,moirai,kernelsynth,ets,norm_mase,97,-9.1%,"[-13.3%, -4.9%]",1.00,0.62
2,moirai,kernelsynth,Mixed,norm_crps,97,-0.2%,"[-3.7%, 3.3%]",0.54,0.48
3,moirai,kernelsynth,Mixed,norm_mase,97,-1.9%,"[-5.5%, 1.4%]",0.85,0.47
4,moirai,kernelsynth,Real Ref,norm_crps,97,-9.9%,"[-15.4%, -4.2%]",1.00,0.62
5,moirai,kernelsynth,Real Ref,norm_mase,97,-8.7%,"[-13.6%, -3.7%]",1.00,0.62
6,chronos,kernelsynth,sde,norm_crps,97,-4.5%,"[-10.8%, 2.1%]",0.91,0.59
7,chronos,kernelsynth,sde,norm_mase,97,0.6%,"[-6.7%, 8.0%]",0.42,0.48
8,chronos,kernelsynth,Mixed,norm_crps,97,3.3%,"[-1.2%, 8.2%]",0.08,0.42
9,chronos,kernelsynth,Mixed,norm_mase,97,10.2%,"[5.5%, 15.2%]",0.00,0.34


In [7]:
# Composition section: 3 seeds only.
composition_rows = []

for model in MODELS:
    for seed in COMPOSITION_SEEDS:
        for condition, label in COMPOSITION_CONDITIONS.items():
            df = load_condition(model, seed, condition, label, "composition")
            if df is not None:
                composition_rows.append(df)

composition_tasks = pd.concat(composition_rows, ignore_index=True)
print(composition_tasks[["model_family", "seed", "condition", "label"]].drop_duplicates().sort_values(["model_family", "condition", "seed"]).to_string(index=False))

model_family  seed       condition    label
     chronos    42     mixed_all11    Mixed
     chronos    43     mixed_all11    Mixed
     chronos    44     mixed_all11    Mixed
     chronos    42 mixed_real_2575    25-75
     chronos    43 mixed_real_2575    25-75
     chronos    44 mixed_real_2575    25-75
     chronos    42 mixed_real_5050    50-50
     chronos    43 mixed_real_5050    50-50
     chronos    44 mixed_real_5050    50-50
     chronos    42 mixed_real_7525    75-25
     chronos    43 mixed_real_7525    75-25
     chronos    44 mixed_real_7525    75-25
     chronos    42  real_reference Real Ref
     chronos    43  real_reference Real Ref
     chronos    44  real_reference Real Ref
      moirai    42     mixed_all11    Mixed
      moirai    43     mixed_all11    Mixed
      moirai    44     mixed_all11    Mixed
      moirai    42 mixed_real_2575    25-75
      moirai    43 mixed_real_2575    25-75
      moirai    44 mixed_real_2575    25-75
      moirai    42 mixed_real_50

In [8]:
# Aggregate each seed first, then report mean/std across seeds.
composition_seed_scores = condition_scores(
    composition_tasks,
    ["model_family", "seed", "condition", "label"],
)

composition_summary = (
    composition_seed_scores
    .groupby(["model_family", "condition", "label"], as_index=False)
    .agg(
        n_seeds=("seed", "nunique"),
        norm_crps_mean=("norm_crps", "mean"),
        norm_crps_std=("norm_crps", "std"),
        norm_mase_mean=("norm_mase", "mean"),
        norm_mase_std=("norm_mase", "std"),
    )
)
composition_summary[["norm_crps_std", "norm_mase_std"]] = composition_summary[["norm_crps_std", "norm_mase_std"]].fillna(0.0)
composition_summary = add_ranks(
    composition_summary,
    group_cols=("model_family",),
    crps_col="norm_crps_mean",
    mase_col="norm_mase_mean",
).sort_values(["model_family", "rank_mean", "norm_crps_mean"])

display(composition_summary)
composition_summary.to_csv(OUTDIR / "composition_3seed_summary.csv", index=False)

,model_family,condition,label,n_seeds,norm_crps_mean,norm_crps_std,norm_mase_mean,norm_mase_std,rank_crps,rank_mase,rank_mean
2,chronos,mixed_real_5050,50-50,3,0.771533,0.016224,1.019267,0.013711,1.0,1.0,1.0
1,chronos,mixed_real_2575,25-75,3,0.783105,0.009357,1.039040,0.011488,3.0,2.0,2.5
4,chronos,real_reference,Real Ref,3,0.779184,0.010485,1.051644,0.008437,2.0,4.0,3.0
3,chronos,mixed_real_7525,75-25,3,0.794117,0.011497,1.051568,0.008858,4.0,3.0,3.5
0,chronos,mixed_all11,Mixed,3,0.889057,0.015373,1.164595,0.005722,5.0,5.0,5.0
8,moirai,mixed_real_7525,75-25,3,0.684828,0.015085,0.984136,0.023206,1.0,1.0,1.0
7,moirai,mixed_real_5050,50-50,3,0.710238,0.015024,1.012407,0.022943,2.0,2.0,2.0
5,moirai,mixed_all11,Mixed,3,0.732825,0.013535,1.058329,0.022260,3.0,3.0,3.0
6,moirai,mixed_real_2575,25-75,3,0.775230,0.011847,1.099658,0.021121,4.0,4.0,4.0
9,moirai,real_reference,Real Ref,3,0.829958,0.013807,1.171832,0.021424,5.0,5.0,5.0


In [9]:
sorting_key = ["Real Ref", "75-25", "50-50", "25-75", "Mixed"]

composition_summary[composition_summary["model_family"] == "chronos"].round(3).sort_values(by="label", key=lambda x: x.map({k: i for i, k in enumerate(sorting_key)}))

,model_family,condition,label,n_seeds,norm_crps_mean,norm_crps_std,norm_mase_mean,norm_mase_std,rank_crps,rank_mase,rank_mean
4,chronos,real_reference,Real Ref,3,0.779,0.010,1.052,0.008,2.0,4.0,3.0
3,chronos,mixed_real_7525,75-25,3,0.794,0.011,1.052,0.009,4.0,3.0,3.5
2,chronos,mixed_real_5050,50-50,3,0.772,0.016,1.019,0.014,1.0,1.0,1.0
1,chronos,mixed_real_2575,25-75,3,0.783,0.009,1.039,0.011,3.0,2.0,2.5
0,chronos,mixed_all11,Mixed,3,0.889,0.015,1.165,0.006,5.0,5.0,5.0


In [10]:
# Composition bootstrap: best mixture ratio vs second-best mixture ratio, Mixed, and Real Ref.
composition_boot_rows = []

for model in MODELS:
    ranked_mix = composition_summary[
        (composition_summary["model_family"] == model) &
        (composition_summary["condition"].isin(MIXTURE_RATIO_CONDITIONS))
    ].sort_values(["rank_mean", "norm_crps_mean"])

    if len(ranked_mix) < 2:
        continue

    best_mix = ranked_mix.iloc[0]["condition"]
    second_mix = ranked_mix.iloc[1]["condition"]

    comparisons = [
        (best_mix, second_mix),
        (best_mix, "mixed_all11"),
        (best_mix, "real_reference"),
    ]

    for condition_a, condition_b in comparisons:
        for metric in ["norm_crps", "norm_mase"]:
            composition_boot_rows.append(
                paired_bootstrap_compare(
                    composition_tasks,
                    model=model,
                    condition_a=condition_a,
                    condition_b=condition_b,
                    metric=metric,
                    pair_cols=("seed", "dataset"),
                    n_boot=10_000,
                    seed=42,
                )
            )

composition_bootstrap = pd.DataFrame(composition_boot_rows)
composition_bootstrap_paper = format_bootstrap_table(composition_bootstrap)

display(composition_bootstrap_paper)
composition_bootstrap.to_csv(OUTDIR / "composition_3seed_bootstrap_raw.csv", index=False)
composition_bootstrap_paper.to_csv(OUTDIR / "composition_3seed_bootstrap_table.csv", index=False)

,model_family,label_a,label_b,metric,n_pairs,Rel. delta,95% CI,P(a better),Win rate
0,moirai,75-25,50-50,norm_crps,291,-3.6%,"[-5.3%, -1.8%]",1.00,0.59
1,moirai,75-25,50-50,norm_mase,291,-2.8%,"[-4.4%, -1.1%]",1.00,0.64
2,moirai,75-25,Mixed,norm_crps,291,-6.6%,"[-8.4%, -4.7%]",1.00,0.71
3,moirai,75-25,Mixed,norm_mase,291,-7.0%,"[-8.9%, -5.2%]",1.00,0.72
4,moirai,75-25,Real Ref,norm_crps,291,-17.5%,"[-20.0%, -15.0%]",1.00,0.88
5,moirai,75-25,Real Ref,norm_mase,291,-16.0%,"[-18.3%, -13.8%]",1.00,0.88
6,chronos,50-50,25-75,norm_crps,291,-1.5%,"[-2.9%, -0.1%]",0.98,0.57
7,chronos,50-50,25-75,norm_mase,291,-1.9%,"[-2.9%, -0.9%]",1.00,0.59
8,chronos,50-50,Mixed,norm_crps,291,-13.2%,"[-15.7%, -10.8%]",1.00,0.80
9,chronos,50-50,Mixed,norm_mase,291,-12.5%,"[-14.7%, -10.3%]",1.00,0.81


In [11]:
def make_single_domain_top5_table(single_tasks, model, top_n=5, include_anchors=True):
    model_df = single_tasks[single_tasks["model_family"] == model].copy()

    generator_df = model_df[model_df["run_type"] == "single_generator"].copy()
    domain_rank = condition_scores(
        generator_df,
        ["model_family", "domain", "run_type", "condition", "label"],
    )
    domain_rank = add_ranks(domain_rank, group_cols=("model_family", "domain"))
    domain_rank = (
        domain_rank
        .sort_values(["domain", "rank_mean", "norm_crps"])
        .groupby("domain", as_index=False)
        .head(top_n)
    )

    if include_anchors:
        anchor_rank = condition_scores(
            model_df[model_df["condition"].isin(["real_reference", "mixed_all11"])],
            ["model_family", "domain", "run_type", "condition", "label"],
        )
        anchor_rank["rank_crps"] = np.nan
        anchor_rank["rank_mase"] = np.nan
        anchor_rank["rank_mean"] = np.nan
        selected = pd.concat([domain_rank, anchor_rank], ignore_index=True)
    else:
        selected = domain_rank

    horizon_scores = condition_scores(
        model_df,
        ["model_family", "domain", "run_type", "condition", "label", "horizon"],
    )

    pivot = horizon_scores.pivot_table(
        index=["model_family", "domain", "run_type", "condition", "label"],
        columns="horizon",
        values=["norm_crps", "norm_mase"],
        aggfunc="first",
    ).reset_index()
    pivot.columns = [
        f"{horizon}_{metric.replace('norm_', '').upper()}" if horizon else metric
        for metric, horizon in pivot.columns.to_flat_index()
    ]

    out = selected.drop(columns=["norm_crps", "norm_mase"]).merge(
        pivot,
        on=["model_family", "domain", "run_type", "condition", "label"],
        how="left",
    )

    for horizon in HORIZON_ORDER:
        for metric in ["CRPS", "MASE"]:
            col = f"{horizon}_{metric}"
            if col not in out.columns:
                out[col] = np.nan
            out[col] = out[col].map(fmt)

    out["sort_rank"] = out["rank_mean"].fillna(99)
    out["condition_order"] = out["label"].map({"Real Ref": 98, "Mixed": 99}).fillna(out["sort_rank"])
    out = out.sort_values(["domain", "condition_order", "sort_rank", "label"])

    return out[[
        "model_family", "domain", "run_type", "label", "condition", "n_tasks",
        "rank_mean", "short_CRPS", "short_MASE", "medium_CRPS", "medium_MASE", "long_CRPS", "long_MASE",
    ]]

In [12]:
single_domain_tables = {}
for model in MODELS:
    table = make_single_domain_top5_table(single_tasks, model, top_n=5, include_anchors=True)
    single_domain_tables[model] = table
    print(f"\nSeed-42 single-generator domain top-5 table: {model}")
    display(table)
    table.to_csv(OUTDIR / f"{model}_seed42_domain_top5_generators_plus_anchors.csv", index=False)


Seed-42 single-generator domain top-5 table: moirai


,model_family,domain,run_type,label,condition,n_tasks,rank_mean,short_CRPS,short_MASE,medium_CRPS,medium_MASE,long_CRPS,long_MASE
0,moirai,Econ/Fin,single_generator,ets,fringe_ets,6,1.0,1.01,1.13,--,--,--,--
1,moirai,Econ/Fin,single_generator,kernelsynth,kernelsynth,6,2.0,1.05,1.14,--,--,--,--
3,moirai,Econ/Fin,single_generator,stepfunction,stepfunction,6,3.5,1.40,1.71,--,--,--,--
2,moirai,Econ/Fin,single_generator,waveform,center_waveform,6,3.5,1.40,1.82,--,--,--,--
4,moirai,Econ/Fin,single_generator,tsi,tsi,6,5.0,1.41,1.82,--,--,--,--
36,moirai,Econ/Fin,anchor,Real Ref,real_reference,6,NaN,1.14,1.49,--,--,--,--
35,moirai,Econ/Fin,anchor,Mixed,mixed_all11,6,NaN,1.23,1.40,--,--,--,--
5,moirai,Energy,single_generator,kernelsynth,kernelsynth,32,1.0,0.87,1.09,0.91,1.31,0.91,1.42
6,moirai,Energy,single_generator,waveform,center_waveform,32,2.0,0.96,1.23,1.08,1.53,1.11,1.70
7,moirai,Energy,single_generator,ets,fringe_ets,32,3.0,0.97,1.17,1.19,1.60,1.25,1.83



Seed-42 single-generator domain top-5 table: chronos


,model_family,domain,run_type,label,condition,n_tasks,rank_mean,short_CRPS,short_MASE,medium_CRPS,medium_MASE,long_CRPS,long_MASE
0,chronos,Econ/Fin,single_generator,kernelsynth,kernelsynth,6,1.0,0.87,0.96,--,--,--,--
1,chronos,Econ/Fin,single_generator,tsi,tsi,6,2.0,1.01,1.14,--,--,--,--
2,chronos,Econ/Fin,single_generator,ets,fringe_ets,6,3.0,1.05,1.22,--,--,--,--
3,chronos,Econ/Fin,single_generator,waveform,center_waveform,6,4.0,1.12,1.31,--,--,--,--
4,chronos,Econ/Fin,single_generator,garch,garch,6,5.5,1.21,1.46,--,--,--,--
36,chronos,Econ/Fin,anchor,Real Ref,real_reference,6,NaN,0.90,1.01,--,--,--,--
35,chronos,Econ/Fin,anchor,Mixed,mixed_all11,6,NaN,0.99,0.95,--,--,--,--
5,chronos,Energy,single_generator,kernelsynth,kernelsynth,32,1.0,0.88,1.11,1.59,2.05,1.71,2.35
7,chronos,Energy,single_generator,sde,sde,32,2.5,1.13,1.32,1.59,1.89,1.45,1.85
6,chronos,Energy,single_generator,stepfunction,stepfunction,32,2.5,1.10,1.29,1.66,2.05,1.37,1.81


In [13]:
metric_cols = ["norm_crps", "norm_mase"]

# domain × horizon × metric winner, ignoring Real Ref
domain_winners = (
    single_tasks
    .query("seed == 42 and condition != 'real_reference'")
    .melt(
        id_vars=["model", "domain", "horizon", "condition"],
        value_vars=metric_cols,
        var_name="metric",
        value_name="score",
    )
    .groupby(["model", "domain", "horizon", "metric", "condition"], as_index=False)
    .agg(score=("score", "mean"))
    .sort_values("score")
    .groupby(["model", "domain", "horizon", "metric"], as_index=False)
    .first()
)

# How many domain-horizon-metric cells Mixed wins
mixed_win_counts = (
    domain_winners
    .assign(is_mixed=lambda d: d["condition"].eq("mixed_all11"))
    .groupby("model", as_index=False)
    .agg(
        mixed_wins=("is_mixed", "sum"),
        total_cells=("is_mixed", "size"),
    )
    .assign(mixed_win_pct=lambda d: 100 * d["mixed_wins"] / d["total_cells"])
)

display(mixed_win_counts)

,model,mixed_wins,total_cells,mixed_win_pct
0,arima,0,30,0.0
1,baseline_fbm,0,30,0.0
2,center_waveform,0,30,0.0
3,chaotic,0,30,0.0
4,fringe_ets,0,30,0.0
5,garch,0,30,0.0
6,kernelsynth,0,30,0.0
7,mixed_all11,30,30,100.0
8,sde,0,30,0.0
9,stepfunction,0,30,0.0


In [14]:
display(
    domain_winners
    .query("condition == 'mixed_all11'")
    .sort_values(["model", "domain", "horizon", "metric"])
)

,model,domain,horizon,metric,condition,score
210,mixed_all11,Econ/Fin,short,norm_crps,mixed_all11,1.192648
211,mixed_all11,Econ/Fin,short,norm_mase,mixed_all11,1.259696
212,mixed_all11,Energy,long,norm_crps,mixed_all11,1.309589
213,mixed_all11,Energy,long,norm_mase,mixed_all11,1.786519
214,mixed_all11,Energy,medium,norm_crps,mixed_all11,1.337681
215,mixed_all11,Energy,medium,norm_mase,mixed_all11,1.727723
216,mixed_all11,Energy,short,norm_crps,mixed_all11,0.893594
217,mixed_all11,Energy,short,norm_mase,mixed_all11,1.103910
218,mixed_all11,Healthcare,short,norm_crps,mixed_all11,0.729059
219,mixed_all11,Healthcare,short,norm_mase,mixed_all11,0.828474


In [15]:
def make_composition_domain_table(composition_tasks, model):
    model_df = composition_tasks[composition_tasks["model_family"] == model].copy()

    seed_domain = condition_scores(
        model_df,
        ["model_family", "seed", "domain", "condition", "label", "horizon"],
    )

    summary = (
        seed_domain
        .groupby(["model_family", "domain", "condition", "label", "horizon"], as_index=False)
        .agg(
            n_seeds=("seed", "nunique"),
            crps_mean=("norm_crps", "mean"),
            crps_std=("norm_crps", "std"),
            mase_mean=("norm_mase", "mean"),
            mase_std=("norm_mase", "std"),
        )
    )
    summary[["crps_std", "mase_std"]] = summary[["crps_std", "mase_std"]].fillna(0.0)

    for metric in ["crps", "mase"]:
        best = summary.groupby(["domain", "horizon"])[f"{metric}_mean"].transform("min")
        summary[f"{metric}_best"] = np.isclose(summary[f"{metric}_mean"], best, rtol=1e-12, atol=1e-12)
        summary[f"{metric}_text"] = summary.apply(
            lambda r: fmt_mean_std(r[f"{metric}_mean"], r[f"{metric}_std"], r[f"{metric}_best"]),
            axis=1,
        )

    pivot = summary.pivot_table(
        index=["model_family", "domain", "label", "condition"],
        columns="horizon",
        values=["crps_text", "mase_text"],
        aggfunc="first",
    ).reset_index()

    pivot.columns = [
        f"{horizon}_{metric.replace('_text', '').upper()}" if horizon else metric
        for metric, horizon in pivot.columns.to_flat_index()
    ]

    for horizon in HORIZON_ORDER:
        for metric in ["CRPS", "MASE"]:
            col = f"{horizon}_{metric}"
            if col not in pivot.columns:
                pivot[col] = "--"
            pivot[col] = pivot[col].fillna("--")

    pivot["ratio_order"] = pd.Categorical(pivot["label"], categories=COMPOSITION_ORDER, ordered=True)
    pivot = pivot.sort_values(["domain", "ratio_order"])

    return pivot[[
        "model_family", "domain", "label", "condition",
        "short_CRPS", "short_MASE", "medium_CRPS", "medium_MASE", "long_CRPS", "long_MASE",
    ]].rename(columns={"label": "R-S Ratio"})

In [16]:
composition_domain_tables = {}
for model in MODELS:
    table = make_composition_domain_table(composition_tasks, model)
    composition_domain_tables[model] = table
    print(f"\n3-seed composition domain table: {model}")
    display(table)
    table.to_csv(OUTDIR / f"{model}_composition_3seed_domain_table.csv", index=False)

print(f"Saved tables to: {OUTDIR}")


3-seed composition domain table: moirai


,model_family,domain,R-S Ratio,condition,short_CRPS,short_MASE,medium_CRPS,medium_MASE,long_CRPS,long_MASE
4,moirai,Econ/Fin,Real Ref,real_reference,1.18 ± 0.03,1.51 ± 0.02,--,--,--,--
2,moirai,Econ/Fin,75-25,mixed_real_7525,1.04 ± 0.13,1.18 ± 0.19,--,--,--,--
1,moirai,Econ/Fin,50-50,mixed_real_5050,★ 0.97 ± 0.03,★ 1.06 ± 0.02,--,--,--,--
0,moirai,Econ/Fin,25-75,mixed_real_2575,0.99 ± 0.06,1.09 ± 0.03,--,--,--,--
3,moirai,Econ/Fin,Mixed,mixed_all11,1.10 ± 0.12,1.24 ± 0.15,--,--,--,--
9,moirai,Energy,Real Ref,real_reference,0.95 ± 0.00,1.16 ± 0.01,1.15 ± 0.02,1.55 ± 0.02,1.31 ± 0.07,1.90 ± 0.12
7,moirai,Energy,75-25,mixed_real_7525,0.80 ± 0.02,0.99 ± 0.01,★ 0.80 ± 0.02,★ 1.15 ± 0.03,★ 0.92 ± 0.09,★ 1.41 ± 0.10
6,moirai,Energy,50-50,mixed_real_5050,★ 0.80 ± 0.01,★ 0.99 ± 0.01,0.92 ± 0.07,1.29 ± 0.10,1.08 ± 0.08,1.60 ± 0.11
5,moirai,Energy,25-75,mixed_real_2575,0.83 ± 0.01,1.03 ± 0.01,1.23 ± 0.06,1.64 ± 0.07,1.31 ± 0.02,1.92 ± 0.08
8,moirai,Energy,Mixed,mixed_all11,0.84 ± 0.01,1.04 ± 0.00,0.94 ± 0.03,1.37 ± 0.04,1.02 ± 0.08,1.60 ± 0.11



3-seed composition domain table: chronos


,model_family,domain,R-S Ratio,condition,short_CRPS,short_MASE,medium_CRPS,medium_MASE,long_CRPS,long_MASE
4,chronos,Econ/Fin,Real Ref,real_reference,0.89 ± 0.01,1.02 ± 0.01,--,--,--,--
2,chronos,Econ/Fin,75-25,mixed_real_7525,0.88 ± 0.03,0.88 ± 0.01,--,--,--,--
1,chronos,Econ/Fin,50-50,mixed_real_5050,0.91 ± 0.03,★ 0.87 ± 0.01,--,--,--,--
0,chronos,Econ/Fin,25-75,mixed_real_2575,★ 0.87 ± 0.03,0.89 ± 0.02,--,--,--,--
3,chronos,Econ/Fin,Mixed,mixed_all11,0.97 ± 0.02,0.98 ± 0.02,--,--,--,--
9,chronos,Energy,Real Ref,real_reference,0.91 ± 0.01,1.14 ± 0.01,1.22 ± 0.03,★ 1.46 ± 0.02,★ 1.12 ± 0.03,★ 1.44 ± 0.03
7,chronos,Energy,75-25,mixed_real_7525,0.84 ± 0.01,1.04 ± 0.02,1.28 ± 0.01,1.57 ± 0.03,1.25 ± 0.05,1.64 ± 0.06
6,chronos,Energy,50-50,mixed_real_5050,★ 0.81 ± 0.01,★ 0.99 ± 0.01,★ 1.21 ± 0.01,1.49 ± 0.02,1.23 ± 0.06,1.58 ± 0.08
5,chronos,Energy,25-75,mixed_real_2575,0.84 ± 0.01,1.05 ± 0.01,1.23 ± 0.02,1.53 ± 0.02,1.21 ± 0.02,1.60 ± 0.02
8,chronos,Energy,Mixed,mixed_all11,0.92 ± 0.02,1.13 ± 0.02,1.62 ± 0.01,1.92 ± 0.02,1.53 ± 0.06,1.94 ± 0.07


Saved tables to: results/fmsd_tables
